In [ ]:
import os
import time
import numpy as np
import pandas as pd
import pymannkendall as mk
from pathlib import Path


In [ ]:
BASE_DIR = Path.cwd()
PROJECT_ROOT = BASE_DIR.parent if BASE_DIR.name == "notebooks" else BASE_DIR

gi_path = os.path.join(PROJECT_ROOT, "outputs", "ihr_gi_star_2001_2025.csv")
gi_long = pd.read_csv(gi_path)  # Cell_ID, Year, Gi_star_z, Gi_star_p_sim, Gi_star_p_norm

gi_wide = gi_long.pivot(index="Cell_ID", columns="Year", values="Gi_star_z")
gi_wide = gi_wide[sorted(gi_wide.columns)]  # ensure chronological 2001 -> 2025

years = list(gi_wide.columns)
print(f"Cells: {len(gi_wide):,} | Years: {years[0]}-{years[-1]} ({len(years)} years)")
print(f"Cells with any missing year: {gi_wide.isna().any(axis=1).sum():,}")

### lag-1 temporal autocorrelation (vectorized across all cells)

In [ ]:
out_dir = os.path.join(PROJECT_ROOT, "outputs", "mann_kendall")
os.makedirs(out_dir, exist_ok=True)

Y = gi_wide.to_numpy(dtype=float)  # (n_cells, 25)
x = Y[:, :-1]
y = Y[:, 1:]

xm = x - x.mean(axis=1, keepdims=True)
ym = y - y.mean(axis=1, keepdims=True)

num = (xm * ym).sum(axis=1)
den = np.sqrt((xm ** 2).sum(axis=1) * (ym ** 2).sum(axis=1))

with np.errstate(invalid="ignore", divide="ignore"):
    lag1_autocorr = np.where(den > 0, num / den, np.nan)

# Rule-of-thumb significance bound for n=25: |r| > 2/sqrt(n)
n = Y.shape[1]
autocorr_threshold = 2 / np.sqrt(n)
autocorr_flag = np.abs(lag1_autocorr) > autocorr_threshold

autocorr_df = pd.DataFrame({
    "Cell_ID": gi_wide.index,
    "Lag1_Autocorr": lag1_autocorr,
    "Autocorr_Significant": autocorr_flag,
})

print(f"Autocorrelation threshold (|r| > {autocorr_threshold:.3f}): "
      f"{autocorr_flag.sum():,} of {len(autocorr_df):,} cells flagged "
      f"({100*autocorr_flag.mean():.1f}%)")

autocorr_path = os.path.join(out_dir, "ihr_lag1_autocorrelation.csv")
autocorr_df.to_csv(autocorr_path, index=False)
print(f"Saved -> {autocorr_path}")

In [ ]:
''' results_path = os.path.join(out_dir, "ihr_mann_kendall_results.csv")
BATCH_SIZE = 5000

autocorr_lookup = autocorr_df.set_index("Cell_ID")["Autocorr_Significant"]

if os.path.exists(results_path):
    done_ids = set(pd.read_csv(results_path, usecols=["Cell_ID"])["Cell_ID"])
    print(f"Resuming: {len(done_ids):,} cells already processed.")
else:
    done_ids = set()
    pd.DataFrame(columns=[
        "Cell_ID", "MK_trend", "MK_h", "MK_p", "MK_z",
        "MK_tau", "MK_s", "MK_var_s", "MK_slope", "Test_Used",
    ]).to_csv(results_path, index=False)

remaining_ids = [cid for cid in gi_wide.index if cid not in done_ids]
print(f"Remaining cells to process: {len(remaining_ids):,}")

t_start = time.time()
buffer = []

for i, cid in enumerate(remaining_ids, start=1):
    series = gi_wide.loc[cid].values

    # Modified (Hamed-Rao) MK for autocorrelated series, ordinary MK otherwise
    use_modified = bool(autocorr_lookup.get(cid, False))
    test_fn = mk.hamed_rao_modification_test if use_modified else mk.original_test

    try:
        result = test_fn(series)
        buffer.append({
            "Cell_ID": cid,
            "MK_trend": result.trend,          # 'increasing' / 'decreasing' / 'no trend'
            "MK_h": result.h,                  # True if trend is significant
            "MK_p": result.p,
            "MK_z": result.z,
            "MK_tau": result.Tau,
            "MK_s": result.s,
            "MK_var_s": result.var_s,
            "MK_slope": result.slope,
            "Test_Used": "modified" if use_modified else "standard",
        })
    except Exception as e:
        buffer.append({
            "Cell_ID": cid, "MK_trend": "error", "MK_h": None, "MK_p": None,
            "MK_z": None, "MK_tau": None, "MK_s": None, "MK_var_s": None,
            "MK_slope": None, "Test_Used": str(e)[:50],
        })

    if len(buffer) >= BATCH_SIZE or i == len(remaining_ids):
        pd.DataFrame(buffer).to_csv(results_path, mode="a", header=False, index=False)
        elapsed = time.time() - t_start
        rate = i / elapsed if elapsed > 0 else 0
        print(f"  {i:,}/{len(remaining_ids):,} done "
              f"({rate:.0f} cells/s, {elapsed:.1f}s elapsed)")
        buffer = []

print(f"\nStep 10 complete -> {results_path}") '''

In [ ]:
results_path = os.path.join(out_dir, "ihr_mann_kendall_results.csv")
BATCH_SIZE = 5000

# Resume if output already exists
if os.path.exists(results_path):
    done_ids = set(pd.read_csv(results_path, usecols=["Cell_ID"])["Cell_ID"])
    print(f"Resuming: {len(done_ids):,} cells already processed.")
else:
    done_ids = set()
    pd.DataFrame(columns=[
        "Cell_ID", "MK_trend", "MK_h", "MK_p", "MK_z",
        "MK_tau", "MK_s", "MK_var_s", "MK_slope", "Test_Used"
    ]).to_csv(results_path, index=False)

remaining_ids = [cid for cid in gi_wide.index if cid not in done_ids]
print(f"Remaining cells to process: {len(remaining_ids):,}")

t_start = time.time()
buffer = []

for i, cid in enumerate(remaining_ids, start=1):
    series = gi_wide.loc[cid].values.astype(float)
    try:
        # Apply ordinary Mann-Kendall to all cells
        result = mk.original_test(series)
        buffer.append({
            "Cell_ID": cid,
            "MK_trend": result.trend,      # increasing / decreasing / no trend
            "MK_h": result.h,              # True if significant
            "MK_p": result.p,
            "MK_z": result.z,
            "MK_tau": result.Tau,
            "MK_s": result.s,
            "MK_var_s": result.var_s,
            "MK_slope": result.slope,
            "Test_Used": "standard"
        })
    except Exception as e:
        buffer.append({
            "Cell_ID": cid,
            "MK_trend": "error",
            "MK_h": None,
            "MK_p": None,
            "MK_z": None,
            "MK_tau": None,
            "MK_s": None,
            "MK_var_s": None,
            "MK_slope": None,
            "Test_Used": str(e)[:50]
        })
    # Save every batch
    if len(buffer) >= BATCH_SIZE or i == len(remaining_ids):
        pd.DataFrame(buffer).to_csv(
            results_path,
            mode="a",
            header=False,
            index=False
        )
        elapsed = time.time() - t_start
        rate = i / elapsed if elapsed > 0 else 0

        print(
            f"  {i:,}/{len(remaining_ids):,} done "
            f"({rate:.0f} cells/s, {elapsed:.1f}s elapsed)"
        )
        buffer = []

print(f"\nStep 10 complete -> {results_path}")

In [ ]:
mk_results = pd.read_csv(results_path)

print(f"Total cells processed: {len(mk_results):,}")
print("\nTrend distribution:")
print(mk_results["MK_trend"].value_counts())

print("\nTest used:")
print(mk_results["Test_Used"].value_counts())

print(f"\nSignificant trends (MK_h == True): {mk_results['MK_h'].sum():,} "
      f"({100*mk_results['MK_h'].mean():.1f}%)")

sig = mk_results[mk_results["MK_h"] == True]
print("\nSignificant trend breakdown:")
print(sig["MK_trend"].value_counts())

In [ ]:
print("NaN MK_z:", mk_results["MK_z"].isna().sum())
print("NaN MK_p:", mk_results["MK_p"].isna().sum())
print("NaN MK_var_s:", mk_results["MK_var_s"].isna().sum())

print("\nNon-positive MK variance:")
print((mk_results["MK_var_s"] <= 0).sum())

print("\nRows with any invalid MK result:")
invalid = (
    mk_results["MK_z"].isna() |
    mk_results["MK_p"].isna() |
    mk_results["MK_var_s"].isna() |
    (mk_results["MK_var_s"] <= 0)
)
print(invalid.sum())

In [ ]:
invalid_ids = mk_results.loc[
    (mk_results["MK_var_s"] <= 0),
    "Cell_ID"
].tolist()

print(f"Invalid cells: {len(invalid_ids):,}")

invalid_series = gi_wide.loc[invalid_ids]

print(invalid_series.shape)
print(invalid_series.head())

In [ ]:
variation = invalid_series.std(axis=1)

print(variation.describe())
print("Zero-variance series:", (variation == 0).sum())

In [ ]:
print(invalid_series.nunique(axis=1).describe())